In [ ]:
import warnings, os, joblib, pickle, json, threading, time
warnings.filterwarnings("ignore")

!pip install pytorch-tabnet
!pip install optuna
import numpy as np
import shutil
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use("Agg")
import seaborn as sns
from collections import Counter
from scipy.stats import f_oneway
from statsmodels.stats.multitest import multipletests

from sklearn.preprocessing import LabelEncoder, RobustScaler
from sklearn.feature_selection import VarianceThreshold
from sklearn.model_selection import train_test_split, GroupKFold
from sklearn.metrics import (
    classification_report, confusion_matrix,
    balanced_accuracy_score, roc_auc_score,
    precision_score, recall_score, f1_score,
)

from imblearn.combine import SMOTETomek
from imblearn.over_sampling import SMOTE

import torch
torch.manual_seed(42)

from pytorch_tabnet.tab_model import TabNetClassifier
import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner
optuna.logging.set_verbosity(optuna.logging.WARNING)

import umap as umap_lib
import shap

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.5/44.5 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 9.7 MB/s eta 0:00:00


1. DATA LOADING & CONFIGURATION

In [ ]:
#Mounting the Google drive

from google.colab import drive

# Unmount Google Drive if it's already mounted or if the session was interrupted
try:
  drive.flush_and_unmount()
  print('Google Drive unmounted successfully.')
except ValueError:
  print('Google Drive was not mounted or could not be unmounted.')
except Exception as e:
  print(f'An unexpected error occurred during unmount: {e}')

# Forcefully remove any existing content in the mount point before mounting
!rm -rf /content/drive/*

drive.mount('/content/drive', force_remount=True)

Drive not mounted, so nothing to flush and unmount.
Google Drive unmounted successfully.
Mounted at /content/drive


In [ ]:
#Configurations
DATA_PATH    = "/content/drive/MyDrive/clean_peak_data.csv"
TARGET_COL   = "arm"
PATIENT_COL  = "id"
META_COLS    = ["filename", "id", "study", "sample_type",
                "date", "casper", "arm", "smoking_status",
                "age", "sex"]
OUTPUT_DIR   = "/content/drive/MyDrive/outputs2"
RANDOM_STATE = 42

NZV_THRESHOLD = 0.01
FDR_ALPHA     = 0.05
MAX_FEATURES  = 400

OUTER_FOLDS         = 5
INNER_FOLDS         = 3
OPTUNA_TRIALS       = 30
TUNE_EPOCHS         = 100
TUNE_PATIENCE       = 15
OUTER_FOLD_EPOCHS   = 150
OUTER_FOLD_PATIENCE = 20
FINAL_EPOCHS        = 200
FINAL_PATIENCE      = 30

RISK_GROUP_MAP = {
    "control"          : "Low risk",
    "stable nodule"    : "Low risk",
    "ipn"              : "Intermediate",
    "cancer possible"  : "Intermediate",
    "cancer"           : "High risk",
}

os.makedirs(OUTPUT_DIR, exist_ok=True)
np.random.seed(RANDOM_STATE)


In [ ]:
#GPU check
print("CUDA available :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU            :", torch.cuda.get_device_name(0))
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ── Keep-alive thread (prevents Colab idle disconnect) ────────────
def _keep_alive():
    i = 0
    while True:
        time.sleep(300)
        i += 1
        print(f"[keep-alive] check {i}", flush=True)

threading.Thread(target=_keep_alive, daemon=True).start()
print("Keep-alive thread started.")




CUDA available : False
Keep-alive thread started.


In [ ]:
# ── Shared helper functions ───────────────────────────────────────

def safe_smote(X_tr, y_tr):
    """SMOTETomek with auto k_neighbors — safe for small minority classes."""
    min_c = int(np.bincount(y_tr).min())
    k     = max(1, min(5, min_c - 1))
    try:
        return SMOTETomek(
            smote=SMOTE(k_neighbors=k, random_state=RANDOM_STATE),
            random_state=RANDOM_STATE
        ).fit_resample(X_tr, y_tr)
    except Exception:
        try:
            return SMOTE(
                k_neighbors=k, random_state=RANDOM_STATE
            ).fit_resample(X_tr, y_tr)
        except Exception:
            return X_tr, y_tr


def make_tabnet(params, verbose=0, max_epochs=100):
    """Build TabNetClassifier from hyperparameter dict."""
    return TabNetClassifier(
        n_d              = params["n_d"],
        n_a              = params["n_d"],
        n_steps          = params["n_steps"],
        gamma            = params["gamma"],
        lambda_sparse    = params["lambda_sparse"],
        optimizer_fn     = torch.optim.Adam,
        optimizer_params = {"lr"          : params["lr"],
                            "weight_decay": params["weight_decay"]},
        scheduler_fn     = torch.optim.lr_scheduler.CosineAnnealingLR,
        scheduler_params = {"T_max": max_epochs, "eta_min": 1e-5},
        mask_type        = "entmax",
        momentum         = 0.02,
        verbose          = verbose,
        seed             = RANDOM_STATE,
        device_name      = DEVICE,
    )



In [ ]:
#Loading the data into notebook
df = pd.read_csv(DATA_PATH)
print(f"  Raw shape          : {df.shape}")
print(f"  Sample type counts :\n{df['sample_type'].value_counts().to_string()}")
print(f"  Arm counts         :\n{df[TARGET_COL].value_counts().to_string()}")
print(f"  Unique patients    : {df[PATIENT_COL].nunique()}")

feat_cols = [c for c in df.columns if c not in META_COLS]
print(f"  m/z feature columns: {len(feat_cols):,}")

df[feat_cols] = df[feat_cols].apply(pd.to_numeric, errors="coerce")
df[feat_cols] = df[feat_cols].replace([np.inf, -np.inf], np.nan)


  Raw shape          : (3264, 11502)
  Sample type counts :
sample_type
RM    1004
EX    1001
RI     631
CP     628
  Arm counts         :
arm
IPN                1158
Control            1032
Stable Nodule       552
Cancer              484
Cancer Possible      31
Cancer possible       4
Cancer                3
  Unique patients    : 1058
  m/z feature columns: 11,492


2. CLEAN LABLES

In [ ]:
print("  Raw unique values:")
for v in sorted(df[TARGET_COL].unique()):
    print(f"    {repr(v):35s}  n={(df[TARGET_COL]==v).sum()}")

df["risk_group"] = (
    df[TARGET_COL]
    .astype(object)      # prevents StringDtype pd.NA issue
    .str.strip()
    .str.lower()
    .map(RISK_GROUP_MAP)
    .astype(object)      # map() reverts to StringDtype in pandas 3.x
)

unmapped = df["risk_group"].isna().sum()
if unmapped > 0:
    print(f"\n  WARNING: {unmapped} rows could not be mapped — dropping.")
    df = df[df["risk_group"].notna()].reset_index(drop=True)

print(f"\n  Cleaned distribution:")
print(df["risk_group"].value_counts().to_string())
print(f"  Total rows: {len(df)}")

# Encode labels
le = LabelEncoder()
y  = le.fit_transform(df["risk_group"].astype(str))
n_classes   = len(le.classes_)
patient_ids = df[PATIENT_COL].values

print(f"\n  Class encoding: {dict(zip(le.classes_, range(n_classes)))}")

  Raw unique values:
    'Cancer'                             n=484
    'Cancer '                            n=3
    'Cancer Possible'                    n=31
    'Cancer possible'                    n=4
    'Control'                            n=1032
    'IPN'                                n=1158
    'Stable Nodule'                      n=552

  Cleaned distribution:
risk_group
Low risk        1584
Intermediate    1193
High risk        487
  Total rows: 3264

  Class encoding: {'High risk': 0, 'Intermediate': 1, 'Low risk': 2}


3. BACKGROUND CORRECTION

In [ ]:
df_ex = df[df["sample_type"] == "EX"].copy().set_index(PATIENT_COL)
df_rm = df[df["sample_type"] == "RM"].copy().set_index(PATIENT_COL)

n_cp_ri = df[df["sample_type"].isin(["CP","RI"])].shape[0]
print(f"  Dropped CP + RI rows : {n_cp_ri}")

# Deduplicate — keep first run per patient per type
n_dup_ex = df_ex.index.duplicated().sum()
n_dup_rm = df_rm.index.duplicated().sum()
print(f"  Duplicate EX removed : {n_dup_ex}")
print(f"  Duplicate RM removed : {n_dup_rm}")
df_ex = df_ex[~df_ex.index.duplicated(keep="first")]
df_rm = df_rm[~df_rm.index.duplicated(keep="first")]

common_pids = df_ex.index.intersection(df_rm.index)
rm_only     = df_rm.index.difference(df_ex.index)
ex_only     = df_ex.index.difference(df_rm.index)

print(f"  Paired EX+RM         : {len(common_pids)}")
print(f"  RM only (kept raw)   : {len(rm_only)}")
print(f"  EX only (dropped)    : {len(ex_only)}")

# EX − RM subtraction
corrected_feats = (
    df_ex.loc[common_pids, feat_cols].values
    - df_rm.loc[common_pids, feat_cols].values
)

meta_keep    = [c for c in META_COLS
                if c in df_ex.columns and c != PATIENT_COL]
corrected_df = pd.DataFrame(corrected_feats,
                             index=common_pids, columns=feat_cols)
corrected_df = pd.concat(
    [df_ex.loc[common_pids, meta_keep + ["risk_group"]],
     corrected_df], axis=1
)
corrected_df.index.name = PATIENT_COL
corrected_df = corrected_df.reset_index()

if len(rm_only) > 0:
    rm_only_df = df_rm.loc[rm_only].reset_index()
    if "risk_group" not in rm_only_df.columns:
        rm_only_df["risk_group"] = (
            rm_only_df[TARGET_COL]
            .astype(object).str.strip().str.lower()
            .map(RISK_GROUP_MAP).astype(object)
        )
    corrected_df = pd.concat(
        [corrected_df, rm_only_df], ignore_index=True
    )

missing_rg = corrected_df["risk_group"].isna().sum()
if missing_rg > 0:
    corrected_df = corrected_df[
        corrected_df["risk_group"].notna()
    ].reset_index(drop=True)

print(f"\n  After correction: {corrected_df.shape}")
print(corrected_df["risk_group"].value_counts().to_string())

# Update y and patient_ids to corrected_df row order
patient_ids = corrected_df[PATIENT_COL].values
y           = le.transform(corrected_df["risk_group"].astype(str))

# NaN → 0 imputation
X_imp = corrected_df[feat_cols].fillna(0).values

# Value split
neg_pct  = (X_imp < 0).sum() / X_imp.size * 100
zero_pct = (X_imp == 0).sum() / X_imp.size * 100
pos_pct  = (X_imp > 0).sum() / X_imp.size * 100
print(f"\n  After imputation → neg:{neg_pct:.1f}%  "
      f"zero:{zero_pct:.1f}%  pos:{pos_pct:.1f}%")

# Log-modulus transform: sign(x) * log2(|x| + 1)
X_lm = np.sign(X_imp) * np.log2(np.abs(X_imp) + 1)
print(f"  X_lm range: [{X_lm.min():.2f}, {X_lm.max():.2f}]")
print(f"  X_lm shape: {X_lm.shape}  (no preprocessing fitted yet)")

  Dropped CP + RI rows : 1259
  Duplicate EX removed : 2
  Duplicate RM removed : 7
  Paired EX+RM         : 996
  RM only (kept raw)   : 1
  EX only (dropped)    : 3

  After correction: (997, 11503)
risk_group
Low risk        534
Intermediate    330
High risk       133

  After imputation → neg:1.7%  zero:96.5%  pos:1.8%
  X_lm range: [-31.73, 31.95]
  X_lm shape: (997, 11492)  (no preprocessing fitted yet)


4. TRAIN-TEST SPLIT

In [ ]:
patient_ids_str = patient_ids.astype(str) # Ensure all patient IDs are strings
unique_pids = np.unique(patient_ids_str)
pid_labels  = np.array([y[patient_ids_str == pid][0]
                        for pid in unique_pids])

train_pids, test_pids = train_test_split(
    unique_pids,
    test_size    = 0.2,
    stratify     = pid_labels,
    random_state = RANDOM_STATE
)

train_mask = np.isin(patient_ids_str, train_pids)
test_mask  = np.isin(patient_ids_str, test_pids)

X_train_raw       = X_lm[train_mask]
X_test_raw        = X_lm[test_mask]
y_train           = y[train_mask]
y_test            = y[test_mask]
train_patient_ids = patient_ids_str[train_mask] # Corrected to use patient_ids_str
test_patient_ids  = patient_ids_str[test_mask]  # Corrected to use patient_ids_str

print(f"  X_train_raw : {X_train_raw.shape}  "
      f"classes: {np.bincount(y_train)}")
print(f"  X_test_raw  : {X_test_raw.shape}   "
      f"classes: {np.bincount(y_test)}")
print(f"  Train patients: {len(train_pids)}")
print(f"  Test  patients: {len(test_pids)}")

np.save(f"{OUTPUT_DIR}/train_patient_ids.npy", train_patient_ids)
np.save(f"{OUTPUT_DIR}/test_patient_ids.npy", test_patient_ids)
np.save(f"{OUTPUT_DIR}/y_train.npy", y_train)
np.save(f"{OUTPUT_DIR}/le_classes.npy", le.classes_)

  X_train_raw : (797, 11492)  classes: [106 264 427]
  X_test_raw  : (200, 11492)   classes: [ 27  66 107]
  Train patients: 797
  Test  patients: 200


5. FIT NZV ON X_train_raw --> TRANSFORM BOTH

In [ ]:
nzv         = VarianceThreshold(threshold=NZV_THRESHOLD)
X_train_nzv = nzv.fit_transform(X_train_raw)   # fit + transform
X_test_nzv  = nzv.transform(X_test_raw)        # transform only

kept_after_nzv = np.array(feat_cols)[nzv.get_support()]

print(f"  X_train_raw  → X_train_nzv : "
      f"{X_train_raw.shape[1]:,} → {X_train_nzv.shape[1]:,} features")
print(f"  X_test_raw   → X_test_nzv  : "
      f"{X_test_raw.shape[1]:,} → {X_test_nzv.shape[1]:,} features")
print(f"  Removed {X_train_raw.shape[1] - X_train_nzv.shape[1]:,} "
      f"near-zero variance features")
print(f"  NZV fitted on X_train_raw only ✓")

joblib.dump(nzv, f"{OUTPUT_DIR}/nzv_filter.pkl")

  X_train_raw  → X_train_nzv : 11,492 → 9,079 features
  X_test_raw   → X_test_nzv  : 11,492 → 9,079 features
  Removed 2,413 near-zero variance features
  NZV fitted on X_train_raw only ✓


['/content/drive/MyDrive/outputs2/nzv_filter.pkl']

6. FIT ROBUSTSCALER ON X_train_nzv --> TRANSFORM BOTH

In [ ]:
scaler         = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train_nzv)   # fit + transform
X_test_scaled  = scaler.transform(X_test_nzv)        # transform only

print(f"  X_train_nzv  → X_train_scaled : {X_train_scaled.shape}")
print(f"  X_test_nzv   → X_test_scaled  : {X_test_scaled.shape}")
print(f"  Scaler fitted on X_train_nzv only ✓")

joblib.dump(scaler, f"{OUTPUT_DIR}/robust_scaler.pkl")

  X_train_nzv  → X_train_scaled : (797, 9079)
  X_test_nzv   → X_test_scaled  : (200, 9079)
  Scaler fitted on X_train_nzv only ✓


['/content/drive/MyDrive/outputs2/robust_scaler.pkl']

7. ANNOVA + FDR ON X_train_scaled ONLY --> annova mask

In [ ]:
groups  = [X_train_scaled[y_train == cls]
           for cls in np.unique(y_train)]
n_feat  = X_train_scaled.shape[1]
f_stats = np.zeros(n_feat)
p_vals  = np.ones(n_feat)

print(f"  Computing F-statistics on {n_feat:,} features "
      f"({X_train_scaled.shape[0]} training samples) ...")

for i in range(n_feat):
    col_groups = [g[:, i] for g in groups]
    try:
        f_stats[i], p_vals[i] = f_oneway(*col_groups)
    except Exception:
        pass

p_vals  = np.nan_to_num(p_vals,  nan=1.0)
f_stats = np.nan_to_num(f_stats, nan=0.0)

rejected, p_fdr, _, _ = multipletests(
    p_vals, alpha=FDR_ALPHA, method="fdr_bh"
)
n_sig = int(rejected.sum())
print(f"  Significant ions (FDR < {FDR_ALPHA}): {n_sig:,} / {n_feat:,}")

if n_sig == 0:
    print("  WARNING: No ions passed FDR — using top 100 by F-stat.")
    top_idx    = np.argsort(f_stats)[::-1][:100]
    anova_mask = np.zeros(n_feat, dtype=bool)
    anova_mask[top_idx] = True
elif n_sig > MAX_FEATURES:
    print(f"  Capping at top {MAX_FEATURES} by F-statistic.")
    sig_idx    = np.where(rejected)[0]
    top_sig    = sig_idx[np.argsort(f_stats[sig_idx])[::-1][:MAX_FEATURES]]
    anova_mask = np.zeros(n_feat, dtype=bool)
    anova_mask[top_sig] = True
else:
    print(f"  Using all {n_sig} significant ions.")
    anova_mask = rejected

# Apply same mask to both — X_test_scaled never influenced anova_mask
X_train_anova = X_train_scaled[:, anova_mask].astype(np.float32)
X_test_anova  = X_test_scaled[:, anova_mask].astype(np.float32)

# Ion names
kept_ions  = kept_after_nzv[anova_mask]
ion_labels = [f"ion_{ion}" for ion in kept_ions]

print(f"\n  X_train_anova : {X_train_anova.shape}")
print(f"  X_test_anova  : {X_test_anova.shape}  ← still locked")
print(f"  Samples/feature: {X_train_anova.shape[0]/X_train_anova.shape[1]:.2f}")
print(f"  ANOVA fitted on X_train_scaled only ✓")

# Save ANOVA artefacts
anova_df = pd.DataFrame({
    "m/z_ion"     : kept_ions,
    "ion_label"   : ion_labels,
    "F_statistic" : f_stats[anova_mask],
    "p_value_raw" : p_vals[anova_mask],
    "p_value_fdr" : p_fdr[anova_mask],
}).sort_values("F_statistic", ascending=False).reset_index(drop=True)
anova_df.to_csv(f"{OUTPUT_DIR}/anova_selected_ions.csv", index=False)

joblib.dump(anova_mask, f"{OUTPUT_DIR}/anova_mask.pkl")
joblib.dump(kept_ions,  f"{OUTPUT_DIR}/selected_ions.pkl")
joblib.dump(le,         f"{OUTPUT_DIR}/label_encoder.pkl")

print(f"\n  Top 10 ions by F-statistic:")
print(anova_df.head(10).to_string(index=False))

np.save(f"{OUTPUT_DIR}/X_test_anova.npy", X_test_anova)
np.save(f"{OUTPUT_DIR}/X_train_anova.npy", X_train_anova)

  Computing F-statistics on 9,079 features (797 training samples) ...
  Significant ions (FDR < 0.05): 422 / 9,079
  Capping at top 400 by F-statistic.

  X_train_anova : (797, 400)
  X_test_anova  : (200, 400)  ← still locked
  Samples/feature: 1.99
  ANOVA fitted on X_train_scaled only ✓

  Top 10 ions by F-statistic:
m/z_ion ion_label  F_statistic  p_value_raw  p_value_fdr
    836   ion_836    77.215057 2.278355e-31 2.068518e-27
   5444  ion_5444    57.090889 6.825652e-24 3.098505e-20
   2546  ion_2546    46.800277 6.114577e-20 1.671473e-16
   4390  ion_4390    46.592460 7.364126e-20 1.671473e-16
   2791  ion_2791    46.145018 1.099310e-19 1.996127e-16
   3508  ion_3508    41.014644 1.119038e-17 1.693291e-14
   4386  ion_4386    38.903625 7.617460e-17 9.754754e-14
    406   ion_406    38.771020 8.595444e-17 9.754754e-14
   6033  ion_6033    38.091550 1.597032e-16 1.611050e-13
   5401  ion_5401    35.488163 1.730069e-15 1.570730e-12


8. NESTED CV ON X_train_annova

In [ ]:
checkpoint_path = f"{OUTPUT_DIR}/nested_cv_checkpoint.pkl"

if os.path.exists(checkpoint_path):
    with open(checkpoint_path, "rb") as f:
        ckpt = pickle.load(f)
    outer_bal_accs    = ckpt["outer_bal_accs"]
    outer_aucs        = ckpt["outer_aucs"]
    outer_best_params = ckpt["outer_best_params"]
    outer_reports     = ckpt["outer_reports"]
    outer_cms         = ckpt["outer_cms"]
    start_fold        = ckpt["completed_folds"]
    print(f"✓ Checkpoint found — resuming from fold "
          f"{start_fold+1}/{OUTER_FOLDS}")
    print(f"  Bal accs so far: "
          f"{[round(x,4) for x in outer_bal_accs]}")
else:
    outer_bal_accs    = []
    outer_aucs        = []
    outer_best_params = []
    outer_reports     = []
    outer_cms         = []
    start_fold        = 0
    print("✓ No checkpoint — starting from fold 1")

# Pre-compute all outer splits
outer_gkf  = GroupKFold(n_splits=OUTER_FOLDS)
all_splits = list(outer_gkf.split(
    X_train_anova, y_train, groups=train_patient_ids.tolist() # Convert to list of strings
))

print(f"\n  GPU     : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"  Input   : X_train_anova {X_train_anova.shape}")
print(f"  Outer   : {OUTER_FOLDS}-fold GroupKFold")
print(f"  Inner   : Optuna {OPTUNA_TRIALS} trials × "
      f"{INNER_FOLDS}-fold (MedianPruner)")

# ── Outer loop ────────────────────────────────────────────────────
for outer_fold in range(start_fold, OUTER_FOLDS):

    outer_tr_idx, outer_val_idx = all_splits[outer_fold]

    print(f"\n{'='*65}")
    print(f"  OUTER FOLD {outer_fold+1} / {OUTER_FOLDS}  "
          f"(X_test_anova still locked)")
    print(f"{'='*65}")

    # Outer split — subsets of X_train_anova
    X_out_tr  = X_train_anova[outer_tr_idx]
    X_out_val = X_train_anova[outer_val_idx]
    y_out_tr  = y_train[outer_tr_idx]
    y_out_val = y_train[outer_val_idx]
    out_tr_pids = train_patient_ids[outer_tr_idx]

    print(f"  outer_train : {X_out_tr.shape}  "
          f"classes: {np.bincount(y_out_tr)}")
    print(f"  outer_val   : {X_out_val.shape}  "
          f"classes: {np.bincount(y_out_val)}")

    torch.cuda.empty_cache()

    # ── Inner Optuna on outer_train ───────────────────────────────
    inner_gkf = GroupKFold(n_splits=INNER_FOLDS)

    def inner_objective(trial):
        params = {
            "n_d"           : trial.suggest_categorical(
                                  "n_d", [8, 16, 32, 64]),
            "n_steps"       : trial.suggest_int("n_steps", 3, 7),
            "gamma"         : trial.suggest_float(
                                  "gamma", 1.0, 2.0, step=0.1),
            "lambda_sparse" : trial.suggest_categorical(
                                  "lambda_sparse",
                                  [1e-4, 1e-3, 1e-2, 1e-1]),
            "lr"            : trial.suggest_categorical(
                                  "lr", [1e-3, 2e-3, 5e-3]),
            "weight_decay"  : trial.suggest_categorical(
                                  "weight_decay",
                                  [1e-5, 1e-4, 1e-3]),
        }

        inner_scores = []

        for fold_num, (in_tr_idx, in_val_idx) in enumerate(
                inner_gkf.split(
                    X_out_tr, y_out_tr,
                    groups=out_tr_pids.tolist())): # Convert to list of strings

            X_in_tr  = X_out_tr[in_tr_idx]
            X_in_val = X_out_tr[in_val_idx]
            y_in_tr  = y_out_tr[in_tr_idx]
            y_in_val = y_out_tr[in_val_idx]

            # SMOTE on inner train only — never on validation
            X_in_tr_b, y_in_tr_b = safe_smote(X_in_tr, y_in_tr)

            clf = make_tabnet(params, verbose=0,
                              max_epochs=TUNE_EPOCHS)
            try:
                clf.fit(
                    X_in_tr_b, y_in_tr_b,
                    eval_set           = [(X_in_val, y_in_val)],
                    eval_name          = ["val"],
                    eval_metric        = ["balanced_accuracy"],
                    max_epochs         = TUNE_EPOCHS,
                    patience           = TUNE_PATIENCE,
                    batch_size         = 64,
                    virtual_batch_size = 32,
                    num_workers        = 0,
                    drop_last          = True,
                )
                score = balanced_accuracy_score(
                    y_in_val, clf.predict(X_in_val)
                )
            except Exception:
                score = 0.0
            finally:
                torch.cuda.empty_cache()

            inner_scores.append(score)

            # Report to pruner — kills bad trials early
            trial.report(float(np.mean(inner_scores)),
                         step=fold_num)
            if trial.should_prune():
                raise optuna.exceptions.TrialPruned()

        return float(np.mean(inner_scores))

    study = optuna.create_study(
        direction = "maximize",
        sampler   = TPESampler(seed=RANDOM_STATE + outer_fold),
        pruner    = MedianPruner(n_startup_trials=5,
                                 n_warmup_steps=1)
    )
    study.optimize(
        inner_objective,
        n_trials          = OPTUNA_TRIALS,
        show_progress_bar = True
    )

    best_params = study.best_params
    outer_best_params.append(best_params)

    pruned   = len([t for t in study.trials
                    if t.state == optuna.trial.TrialState.PRUNED])
    complete = len([t for t in study.trials
                    if t.state == optuna.trial.TrialState.COMPLETE])

    print(f"\n  Inner best  : {study.best_value:.4f}")
    print(f"  Complete: {complete}  Pruned: {pruned}  "
          f"(saved ~{pruned*INNER_FOLDS} training runs)")
    print(f"  Best params : {best_params}")

    study.trials_dataframe().to_csv(
        f"{OUTPUT_DIR}/optuna_outer_fold{outer_fold+1}.csv",
        index=False
    )

    # ── Retrain on full outer_train with best params + SMOTE ──────
    print(f"\n  Retraining on full outer_train ...")
    torch.cuda.empty_cache()

    X_out_tr_b, y_out_tr_b = safe_smote(X_out_tr, y_out_tr)
    print(f"  After SMOTE : {np.bincount(y_out_tr_b)}")

    fold_model = make_tabnet(best_params, verbose=5,
                             max_epochs=OUTER_FOLD_EPOCHS)
    fold_model.fit(
        X_out_tr_b, y_out_tr_b,
        eval_set           = [(X_out_val, y_out_val)],
        eval_name          = ["outer_val"],
        eval_metric        = ["balanced_accuracy"],
        max_epochs         = OUTER_FOLD_EPOCHS,
        patience           = OUTER_FOLD_PATIENCE,
        batch_size         = 64,
        virtual_batch_size = 32,
        num_workers        = 0,
        drop_last          = True,
    )
    torch.cuda.empty_cache()

    # ── Evaluate on outer_val ─────────────────────────────────────
    y_val_pred = fold_model.predict(X_out_val)
    y_val_prob = fold_model.predict_proba(X_out_val)

    ba = balanced_accuracy_score(y_out_val, y_val_pred)
    try:
        au = roc_auc_score(y_out_val, y_val_prob,
                           multi_class="ovr", average="macro")
    except Exception:
        au = float("nan")

    outer_bal_accs.append(ba)
    outer_aucs.append(au)
    outer_reports.append(
        classification_report(
            y_out_val, y_val_pred,
            target_names=le.classes_,
            output_dict=True
        )
    )
    outer_cms.append(confusion_matrix(y_out_val, y_val_pred))

    fold_model.save_model(
        f"{OUTPUT_DIR}/tabnet_outer_fold{outer_fold+1}"
    )

    # Save fold predictions for ROC curve computation
    pd.DataFrame({
        "y_true": y_out_val,
        "y_pred": y_val_pred,
    }).to_csv(
        f"{OUTPUT_DIR}/fold{outer_fold+1}_predictions.csv",
        index=False
    )
    pd.DataFrame(
        y_val_prob,
        columns=[f"prob_{c}" for c in le.classes_]
    ).to_csv(
        f"{OUTPUT_DIR}/fold{outer_fold+1}_probabilities.csv",
        index=False
    )

    print(f"\n  ── Outer Fold {outer_fold+1} Result ──────────────")
    print(f"  Balanced Acc : {ba:.4f}")
    print(f"  AUC (macro)  : {au:.4f}")
    print(classification_report(
        y_out_val, y_val_pred, target_names=le.classes_))

    # ── Save checkpoint immediately after fold completes ──────────
    checkpoint = {
        "outer_bal_accs"    : outer_bal_accs,
        "outer_aucs"        : outer_aucs,
        "outer_best_params" : outer_best_params,
        "outer_reports"     : outer_reports,
        "outer_cms"         : outer_cms,
        "completed_folds"   : outer_fold + 1,
    }
    with open(checkpoint_path, "wb") as f:
        pickle.dump(checkpoint, f)

    # CSV copies for easy inspection
    joblib.dump(outer_reports,
                f"{OUTPUT_DIR}/outer_reports_so_far.pkl")
    joblib.dump(outer_cms,
                f"{OUTPUT_DIR}/outer_confusion_matrices_so_far.pkl")
    joblib.dump(outer_best_params,
                f"{OUTPUT_DIR}/outer_best_params_so_far.pkl")
    pd.DataFrame({
        "fold"             : list(range(1, len(outer_bal_accs)+1)),
        "balanced_accuracy": outer_bal_accs,
        "auc"              : outer_aucs,
    }).to_csv(f"{OUTPUT_DIR}/outer_metrics_so_far.csv", index=False)

    print(f"  ✓ Checkpoint saved — fold {outer_fold+1} complete")
    print(f"  Running mean : {np.mean(outer_bal_accs):.4f}")

np.save(f"{OUTPUT_DIR}/outer_bal_accs.npy", np.array(outer_bal_accs))
np.save(f"{OUTPUT_DIR}/outer_aucs.npy", np.array(outer_aucs))
np.save(f"{OUTPUT_DIR}/outer_cms.npy", np.array(outer_cms))

with open(f"{OUTPUT_DIR}/outer_reports.pkl", "wb") as f:
    pickle.dump(outer_reports, f)
# ── Nested CV Summary ─────────────────────────────────────────────
print(f"\n{'='*65}")
print("  NESTED CV COMPLETE")
print(f"{'='*65}")

print(f"\n  Per-fold:")
for i, (ba, au) in enumerate(zip(outer_bal_accs, outer_aucs)):
    print(f"    Fold {i+1}  BA:{ba:.4f}  AUC:{au:.4f}  "
          f"n_d={outer_best_params[i]['n_d']}  "
          f"n_steps={outer_best_params[i]['n_steps']}  "
          f"γ={outer_best_params[i]['gamma']:.1f}  "
          f"λ={outer_best_params[i]['lambda_sparse']}")

print(f"\n  Nested CV Balanced Acc : "
      f"{np.mean(outer_bal_accs):.4f} "
      f"± {np.std(outer_bal_accs):.4f}")
print(f"  Nested CV AUC (macro)  : "
      f"{np.nanmean(outer_aucs):.4f} "
      f"± {np.nanstd(outer_aucs):.4f}")

print(f"\n  Per-class (mean ± std across {OUTER_FOLDS} folds):")
for cls in le.classes_:
    p  = [r[cls]["precision"] for r in outer_reports]
    r_ = [r[cls]["recall"]    for r in outer_reports]
    f1 = [r[cls]["f1-score"]  for r in outer_reports]
    print(f"    {cls:15s}  "
          f"P:{np.mean(p):.3f}±{np.std(p):.3f}  "
          f"R:{np.mean(r_):.3f}±{np.std(r_):.3f}  "
          f"F1:{np.mean(f1):.3f}±{np.std(f1):.3f}")

# Most common best hyperparameters → final model
most_common_params = {}
for key in outer_best_params[0].keys():
    vals = [bp[key] for bp in outer_best_params]
    most_common_params[key] = Counter(vals).most_common(1)[0][0]

print(f"\n  Most common params → final model:")
for k, v in most_common_params.items():
    print(f"    {k:20s} : {v}")

with open(f"{OUTPUT_DIR}/most_common_params.json", "w") as f:
    json.dump(most_common_params, f, indent=2)


✓ No checkpoint — starting from fold 1

  GPU     : CPU
  Input   : X_train_anova (797, 400)
  Outer   : 5-fold GroupKFold
  Inner   : Optuna 30 trials × 3-fold (MedianPruner)

  OUTER FOLD 1 / 5  (X_test_anova still locked)
  outer_train : (637, 400)  classes: [ 87 215 335]
  outer_val   : (160, 400)  classes: [19 49 92]


  0%|          | 0/30 [00:00<?, ?it/s]


Early stopping occurred at epoch 69 with best_epoch = 54 and best_val_balanced_accuracy = 0.53826

Early stopping occurred at epoch 39 with best_epoch = 24 and best_val_balanced_accuracy = 0.57452

Early stopping occurred at epoch 41 with best_epoch = 26 and best_val_balanced_accuracy = 0.55801

Early stopping occurred at epoch 52 with best_epoch = 37 and best_val_balanced_accuracy = 0.65615

Early stopping occurred at epoch 90 with best_epoch = 75 and best_val_balanced_accuracy = 0.69244

Early stopping occurred at epoch 62 with best_epoch = 47 and best_val_balanced_accuracy = 0.69129
[keep-alive] check 1

Early stopping occurred at epoch 38 with best_epoch = 23 and best_val_balanced_accuracy = 0.49887

Early stopping occurred at epoch 90 with best_epoch = 75 and best_val_balanced_accuracy = 0.6033

Early stopping occurred at epoch 66 with best_epoch = 51 and best_val_balanced_accuracy = 0.56154

Early stopping occurred at epoch 66 with best_epoch = 51 and best_val_balanced_accuracy 

  0%|          | 0/30 [00:00<?, ?it/s]


Early stopping occurred at epoch 29 with best_epoch = 14 and best_val_balanced_accuracy = 0.40873

Early stopping occurred at epoch 15 with best_epoch = 0 and best_val_balanced_accuracy = 0.45737
[keep-alive] check 13

Early stopping occurred at epoch 32 with best_epoch = 17 and best_val_balanced_accuracy = 0.47812

Early stopping occurred at epoch 34 with best_epoch = 19 and best_val_balanced_accuracy = 0.3814

Early stopping occurred at epoch 40 with best_epoch = 25 and best_val_balanced_accuracy = 0.39984

Early stopping occurred at epoch 26 with best_epoch = 11 and best_val_balanced_accuracy = 0.384

Early stopping occurred at epoch 31 with best_epoch = 16 and best_val_balanced_accuracy = 0.40666

Early stopping occurred at epoch 32 with best_epoch = 17 and best_val_balanced_accuracy = 0.40452

Early stopping occurred at epoch 68 with best_epoch = 53 and best_val_balanced_accuracy = 0.41534

Early stopping occurred at epoch 43 with best_epoch = 28 and best_val_balanced_accuracy = 

...


Early stopping occurred at epoch 17 with best_epoch = 2 and best_val_balanced_accuracy = 0.44519

Early stopping occurred at epoch 49 with best_epoch = 34 and best_val_balanced_accuracy = 0.40283

Early stopping occurred at epoch 49 with best_epoch = 34 and best_val_balanced_accuracy = 0.44521

Early stopping occurred at epoch 48 with best_epoch = 33 and best_val_balanced_accuracy = 0.43927

Early stopping occurred at epoch 79 with best_epoch = 64 and best_val_balanced_accuracy = 0.43017

Early stopping occurred at epoch 46 with best_epoch = 31 and best_val_balanced_accuracy = 0.4037
[keep-alive] check 24

Early stopping occurred at epoch 16 with best_epoch = 1 and best_val_balanced_accuracy = 0.3762

Early stopping occurred at epoch 20 with best_epoch = 5 and best_val_balanced_accuracy = 0.38228

Early stopping occurred at epoch 15 with best_epoch = 0 and best_val_balanced_accuracy = 0.37011

Early stopping occurred at epoch 53 with best_epoch = 38 and best_val_balanced_accuracy = 0.

  0%|          | 0/30 [00:00<?, ?it/s]


Early stopping occurred at epoch 53 with best_epoch = 38 and best_val_balanced_accuracy = 0.60712

Early stopping occurred at epoch 64 with best_epoch = 49 and best_val_balanced_accuracy = 0.67763

Early stopping occurred at epoch 86 with best_epoch = 71 and best_val_balanced_accuracy = 0.73543

Early stopping occurred at epoch 80 with best_epoch = 65 and best_val_balanced_accuracy = 0.69219
[keep-alive] check 37

Early stopping occurred at epoch 57 with best_epoch = 42 and best_val_balanced_accuracy = 0.68145

Early stopping occurred at epoch 76 with best_epoch = 61 and best_val_balanced_accuracy = 0.69161

Early stopping occurred at epoch 41 with best_epoch = 26 and best_val_balanced_accuracy = 0.43408

Early stopping occurred at epoch 62 with best_epoch = 47 and best_val_balanced_accuracy = 0.41416

Early stopping occurred at epoch 38 with best_epoch = 23 and best_val_balanced_accuracy = 0.44466

Early stopping occurred at epoch 78 with best_epoch = 63 and best_val_balanced_accurac

  0%|          | 0/30 [00:00<?, ?it/s]


Early stopping occurred at epoch 22 with best_epoch = 7 and best_val_balanced_accuracy = 0.43271

Early stopping occurred at epoch 57 with best_epoch = 42 and best_val_balanced_accuracy = 0.40736

Early stopping occurred at epoch 20 with best_epoch = 5 and best_val_balanced_accuracy = 0.4454

Early stopping occurred at epoch 43 with best_epoch = 28 and best_val_balanced_accuracy = 0.50479

Early stopping occurred at epoch 19 with best_epoch = 4 and best_val_balanced_accuracy = 0.40844

Early stopping occurred at epoch 19 with best_epoch = 4 and best_val_balanced_accuracy = 0.41915
[keep-alive] check 46

Early stopping occurred at epoch 44 with best_epoch = 29 and best_val_balanced_accuracy = 0.53899

Early stopping occurred at epoch 34 with best_epoch = 19 and best_val_balanced_accuracy = 0.52725

Early stopping occurred at epoch 21 with best_epoch = 6 and best_val_balanced_accuracy = 0.52397

Early stopping occurred at epoch 81 with best_epoch = 66 and best_val_balanced_accuracy = 0.

9. FINAL MODEL ON FULL X_train annova + SMOTE

In [ ]:
# ════════════════════════════════════════════════════════════════════
# CELL 10 — FINAL MODEL ON FULL X_train_anova + SMOTE
# ════════════════════════════════════════════════════════════════════
X_train_bal, y_train_bal = safe_smote(X_train_anova, y_train)

X_tr_fit, X_tr_mon, y_tr_fit, y_tr_mon = train_test_split(
    X_train_bal, y_train_bal,
    test_size=0.1, stratify=y_train_bal,
    random_state=RANDOM_STATE
)

tabnet_final = make_tabnet(
    most_common_params,
    verbose=10,
    max_epochs=FINAL_EPOCHS
)

tabnet_final.fit(
    X_tr_fit, y_tr_fit,
    eval_set=[(X_tr_mon, y_tr_mon)],
    eval_name=["train_monitor"],
    eval_metric=["balanced_accuracy"],
    max_epochs=FINAL_EPOCHS,
    patience=FINAL_PATIENCE,
    batch_size=64,
    virtual_batch_size=32,
    num_workers=0,
    drop_last=True,
)

tabnet_final.save_model(f"{OUTPUT_DIR}/tabnet_final_model")
print(f"\n  Final model saved → tabnet_final_model2.zip")



epoch 0  | loss: 1.96992 | train_monitor_balanced_accuracy: 0.38852 |  0:00:01s
epoch 10 | loss: 0.90962 | train_monitor_balanced_accuracy: 0.73994 |  0:00:12s
epoch 20 | loss: 0.38047 | train_monitor_balanced_accuracy: 0.85825 |  0:00:24s
epoch 30 | loss: 0.22964 | train_monitor_balanced_accuracy: 0.9055  |  0:00:35s
epoch 40 | loss: 0.13129 | train_monitor_balanced_accuracy: 0.93706 |  0:00:45s
epoch 50 | loss: 0.14259 | train_monitor_balanced_accuracy: 0.93706 |  0:00:56s
epoch 60 | loss: 0.08629 | train_monitor_balanced_accuracy: 0.94463 |  0:01:08s

Early stopping occurred at epoch 67 with best_epoch = 37 and best_train_monitor_balanced_accuracy = 0.96087
Successfully saved model at /content/drive/MyDrive/outputs2/tabnet_final_model.zip

  Final model saved → tabnet_final_model2.zip


10. EVALUATE ONCE ON X_test_annova

In [ ]:
print("  CELL 11: EVALUATE ON X_test_anova — FIRST AND ONLY TIME")
print(f"{'='*65}")

y_pred = tabnet_final.predict(X_test_anova)
y_prob = tabnet_final.predict_proba(X_test_anova)

# Core metrics
bal_acc        = balanced_accuracy_score(y_test, y_pred)
try:
    auc_score  = roc_auc_score(
        y_test, y_prob, multi_class="ovr", average="macro"
    )
except Exception:
    auc_score  = float("nan")

prec_macro     = precision_score(y_test, y_pred, average="macro",
                                  zero_division=0)
rec_macro      = recall_score(y_test, y_pred, average="macro",
                               zero_division=0)
f1_macro       = f1_score(y_test, y_pred, average="macro",
                           zero_division=0)

# Per-class specificity
cm = confusion_matrix(y_test, y_pred)
specificities = []
for i in range(len(cm)):
    TP = cm[i, i]
    FN = cm[i, :].sum() - TP
    FP = cm[:, i].sum() - TP
    TN = cm.sum() - (TP + FN + FP)
    spec = TN / (TN + FP) if (TN + FP) > 0 else 0.0
    specificities.append(spec)

print(f"\n  ── Nested CV (X_train_anova, {OUTER_FOLDS}-fold) ────────")
print(f"  Balanced Acc : {np.mean(outer_bal_accs):.4f} "
      f"± {np.std(outer_bal_accs):.4f}")
print(f"  AUC (macro)  : {np.nanmean(outer_aucs):.4f} "
      f"± {np.nanstd(outer_aucs):.4f}")

print(f"\n  ── Final Test (X_test_anova, 20% holdout) ───────────")
print(f"  Balanced Acc      : {bal_acc:.4f}")
print(f"  AUC (macro OvR)   : {auc_score:.4f}")
print(f"  Precision (macro) : {prec_macro:.4f}")
print(f"  Recall (macro)    : {rec_macro:.4f}")
print(f"  F1 (macro)        : {f1_macro:.4f}")
print(f"\n  Per-class specificity:")
for cls, spec in zip(le.classes_, specificities):
    print(f"    {cls:15s} : {spec:.4f}")
print(f"  Mean specificity  : {np.mean(specificities):.4f}")

gap = np.mean(outer_bal_accs) - bal_acc
print(f"\n  Generalisation gap (CV − test) : {gap:+.4f}")
if abs(gap) < 0.05:
    print("  ✓ Good generalisation (gap < 0.05)")
elif gap > 0.05:
    print("  ⚠ CV > test — check for overfitting")
else:
    print("  ℹ Test > CV — lucky split or conservative CV")

print(f"\n  Classification Report (X_test_anova):")
print(classification_report(y_test, y_pred,
                            target_names=le.classes_))
np.save(f"{OUTPUT_DIR}/y_test.npy", y_test)
np.save(f"{OUTPUT_DIR}/y_test_pred.npy", y_pred)
np.save(f"{OUTPUT_DIR}/y_test_prob.npy", y_prob)

# Save SHAP inputs
np.save(f"{OUTPUT_DIR}/X_train_anova.npy", X_train_anova)
np.save(f"{OUTPUT_DIR}/X_test_anova.npy", X_test_anova)

# Save feature labels
joblib.dump(ion_labels, f"{OUTPUT_DIR}/ion_labels.pkl")


  CELL 11: EVALUATE ON X_test_anova — FIRST AND ONLY TIME

  ── Nested CV (X_train_anova, 5-fold) ────────
  Balanced Acc : 0.7451 ± 0.0279
  AUC (macro)  : 0.8615 ± 0.0224

  ── Final Test (X_test_anova, 20% holdout) ───────────
  Balanced Acc      : 0.6566
  AUC (macro OvR)   : 0.8424
  Precision (macro) : 0.7122
  Recall (macro)    : 0.6566
  F1 (macro)        : 0.6773

  Per-class specificity:
    High risk       : 0.9769
    Intermediate    : 0.7985
    Low risk        : 0.6774
  Mean specificity  : 0.8176

  Generalisation gap (CV − test) : +0.0884
  ⚠ CV > test — check for overfitting

  Classification Report (X_test_anova):
              precision    recall  f1-score   support

   High risk       0.79      0.56      0.65        27
Intermediate       0.62      0.67      0.64        66
    Low risk       0.73      0.75      0.74       107

    accuracy                           0.69       200
   macro avg       0.71      0.66      0.68       200
weighted avg       0.70      0.69 

11. SHAP ON FINAL MODEL

In [ ]:
def tabnet_proba(X):
    # This function is not used directly by explainer after modification
    return tabnet_final.predict_proba(X.astype(np.float32))

bg_n  = min(150, X_train_anova.shape[0])
exp_n = min(100, X_test_anova.shape[0])
bg    = shap.sample(X_train_anova, bg_n, random_state=RANDOM_STATE)
X_exp = X_test_anova[:exp_n]

print(f"  Background : {bg_n} X_train_anova samples")
print(f"  Explain    : {exp_n} X_test_anova samples")
print("  This may take 5–15 min on CPU ...")

all_shap_values = []
class_names = list(le.classes_)

for cls_idx, cls_name in enumerate(class_names):
    # Define a function for the explainer that returns the probability of the current class
    # Use a lambda to capture cls_idx correctly in the loop
    explainer_single_output = shap.KernelExplainer(
        lambda x, c_idx=cls_idx: tabnet_final.predict_proba(x.astype(np.float32))[:, c_idx],
        bg
    )
    # Calculate SHAP values for this specific class
    # This will now produce a 2D array of shape (exp_n, n_features)
    sv_for_class = explainer_single_output.shap_values(X_exp, nsamples=300)
    all_shap_values.append(sv_for_class)

# Now, the rest of the code can iterate through all_shap_values

for cls_idx, cls_name in enumerate(class_names):
    sv      = all_shap_values[cls_idx] # This sv will have shape (exp_n, n_features)
    safe    = cls_name.lower().replace(" ", "_")

    plt.figure(figsize=(10, 7))
    shap.summary_plot(sv, X_exp, feature_names=ion_labels,
                      show=False, plot_type="dot", max_display=20)
    plt.title(f"SHAP — Top 20 ions driving [{cls_name}]", pad=6)
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/shap_beeswarm_{safe}.png",
                dpi=300, bbox_inches="tight")
    plt.savefig(f"{OUTPUT_DIR}/shap_beeswarm_{safe}.pdf",
                dpi=300, bbox_inches="tight")
    plt.close()

    plt.figure(figsize=(10, 5))
    shap.summary_plot(sv, X_exp, feature_names=ion_labels,
                      show=False, plot_type="bar", max_display=20)
    plt.title(f"Mean |SHAP| — [{cls_name}] biomarker ions", pad=6)
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/shap_bar_{safe}.png",
                dpi=300, bbox_inches="tight")
    plt.savefig(f"{OUTPUT_DIR}/shap_bar_{safe}.pdf",
                dpi=300, bbox_inches="tight")
    plt.close()

    mean_shap = np.abs(sv).mean(axis=0)
    pd.DataFrame({
        "m/z_ion"       : kept_ions,
        "ion_label"     : ion_labels,
        "mean_abs_shap" : mean_shap,
    }).sort_values("mean_abs_shap", ascending=False
    ).reset_index(drop=True
    ).to_csv(f"{OUTPUT_DIR}/biomarker_ions_{safe}.csv", index=False)

    print(f"  [{cls_name}] SHAP saved.")

np.save(f"{OUTPUT_DIR}/shap_values_all_classes.npy",
        np.array(all_shap_values, dtype=object))

# Save final results summary
results = pd.DataFrame([
    {
        "n_train_anova"             : X_train_anova.shape[0],
        "n_test_anova"              : X_test_anova.shape[0],
        "n_features"                : X_train_anova.shape[1],
        "nested_cv_ba_mean"         : round(np.mean(outer_bal_accs), 4),
        "nested_cv_ba_std"          : round(np.std(outer_bal_accs), 4),
        "nested_cv_auc_mean"        : round(np.nanmean(outer_aucs), 4),
        "nested_cv_auc_std"         : round(np.nanstd(outer_aucs), 4),
        "test_balanced_acc"         : round(bal_acc, 4),
        "test_auc_macro"            : round(auc_score, 4),
        "test_precision_macro"      : round(prec_macro, 4),
        "test_recall_macro"         : round(rec_macro, 4),
        "test_f1_macro"             : round(f1_macro, 4),
        "test_mean_specificity"     : round(np.mean(specificities), 4),
        "generalisation_gap"        : round(gap, 4),
        **{f"param_{k}": v for k, v in most_common_params.items()},
        "classes"                   : " | ".join(le.classes_),
        "bg_correction"             : "EX - RM (negatives kept)",
        "transform"                 : "sign(x)*log2(|x|+1)",
        "leakage"                   : "None — NZV/Scaler/ANOVA "
                                      "fitted on X_train only",
    }
])
results.to_csv(f"{OUTPUT_DIR}/final_results_summary.csv", index=False)

  Background : 150 X_train_anova samples
  Explain    : 100 X_test_anova samples
  This may take 5–15 min on CPU ...


  0%|          | 0/100 [00:00<?, ?it/s]

[keep-alive] check 58
[keep-alive] check 59
[keep-alive] check 60
[keep-alive] check 61
[keep-alive] check 62
[keep-alive] check 63
[keep-alive] check 64


  0%|          | 0/100 [00:00<?, ?it/s]

[keep-alive] check 65
[keep-alive] check 66
[keep-alive] check 67
[keep-alive] check 68
[keep-alive] check 69
[keep-alive] check 70


  0%|          | 0/100 [00:00<?, ?it/s]

[keep-alive] check 71
[keep-alive] check 72
[keep-alive] check 73
[keep-alive] check 74
[keep-alive] check 75
[keep-alive] check 76
[keep-alive] check 77
  [High risk] SHAP saved.
  [Intermediate] SHAP saved.
  [Low risk] SHAP saved.
